In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("..")
SILVER_DIR = BASE / "silver_data"
REPORT_DIR = BASE / "reports"
SILVER_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv("../orders_enriched.csv",
                  usecols=["sales_employee_id", "sales_employee_name",
                           "marital_status", "education_level", "years_experience"])
print("=== PROFILING SALES_EMPLOYEE (raw order-level) ===")
print("Shape:", df.shape)
print("Duplicate sales_employee_id rows:", df["sales_employee_id"].duplicated().sum())

=== PROFILING SALES_EMPLOYEE (raw order-level) ===
Shape: (646945, 5)
Duplicate sales_employee_id rows: 646745


In [ ]:
check = df.groupby("sales_employee_id")[["sales_employee_name", "marital_status",
                                          "education_level", "years_experience"]].nunique()
inconsistent = check[(check > 1).any(axis=1)]
print("Số nhân viên có thuộc tính KHÔNG hằng định qua các đơn hàng:", len(inconsistent))
if len(inconsistent) > 0:
    inconsistent.to_csv(REPORT_DIR / "_review_sales_employee_inconsistent.csv")
    raise ValueError("Có nhân viên dữ liệu không hằng định - cần xử lý thủ công trước khi dedupe.")

df = df.drop_duplicates(subset="sales_employee_id").rename(
    columns={"sales_employee_name": "name"}).reset_index(drop=True)
print("Sau dedupe:", df.shape)

Số nhân viên có thuộc tính KHÔNG hằng định qua các đơn hàng: 0
Sau dedupe: (200, 5)


## 1. Missing Data

In [ ]:
bad_pk = df[df["sales_employee_id"].isna()].copy()
if len(bad_pk):
    bad_pk.to_csv(REPORT_DIR / "_rejected_sales_employee_missing_id.csv", index=False)
    df = df[df["sales_employee_id"].notna()].copy()

print("Missing by column:\n", df.isna().sum())

Missing by column:
 sales_employee_id    0
name                 0
marital_status       0
education_level      0
years_experience     0
dtype: int64


## 2. Outlier / Domain rule

In [ ]:
df["years_experience"] = pd.to_numeric(df["years_experience"], errors="coerce")
invalid_exp = df[df["years_experience"] < 0].copy()
if len(invalid_exp):
    invalid_exp.to_csv(REPORT_DIR / "_rejected_sales_employee_invalid_experience.csv", index=False)
df.loc[df["years_experience"] < 0, "years_experience"] = np.nan
print("Invalid years_experience:", len(invalid_exp))

Invalid years_experience: 0


## 3. Inconsistency + kiểu dữ liệu

In [ ]:
for col in ["sales_employee_id", "name", "marital_status", "education_level"]:
    df[col] = df[col].astype("string").str.strip()
df["years_experience"] = df["years_experience"].astype("Int64")
print(df.dtypes)

sales_employee_id    string
name                 string
marital_status       string
education_level      string
years_experience      Int64
dtype: object


## 4. Missing sau xử lý

In [ ]:
if df["years_experience"].isna().any():
    med = df["years_experience"].median()
    if pd.isna(med):
        raise ValueError("years_experience: không có median để impute")
    df["years_experience"] = df["years_experience"].fillna(round(med)).astype("Int64")

print("Missing after processing:\n", df.isna().sum())

Missing after processing:
 sales_employee_id    0
name                 0
marital_status       0
education_level      0
years_experience     0
dtype: int64


## 5. Validation

In [ ]:
assert df["sales_employee_id"].notna().all(), "SALES_EMPLOYEE: ID còn thiếu"
assert df["sales_employee_id"].is_unique, "SALES_EMPLOYEE: ID bị trùng"
assert df["name"].notna().all(), "SALES_EMPLOYEE: name còn thiếu"
assert df["marital_status"].notna().all(), "SALES_EMPLOYEE: marital_status còn thiếu"
assert df["education_level"].notna().all(), "SALES_EMPLOYEE: education_level còn thiếu"
assert df["years_experience"].notna().all(), "SALES_EMPLOYEE: years_experience còn thiếu"
assert (df["years_experience"] >= 0).all(), "SALES_EMPLOYEE: years_experience âm"
print("SALES_EMPLOYEE: validation đạt yêu cầu Silver.")

SALES_EMPLOYEE: validation đạt yêu cầu Silver.


In [ ]:
df.to_csv(SILVER_DIR / "SALES_EMPLOYEE.csv", index=False)
log = [
    ["SALES_EMPLOYEE", "Dedupe", len(df), "Gộp 646,945 dòng order về 200 nhân viên duy nhất, đã kiểm chứng hằng định"],
    ["SALES_EMPLOYEE", "Missing/keys", len(df), "Không bịa sales_employee_id"],
    ["SALES_EMPLOYEE", "Domain rule", len(df), "years_experience âm được cô lập và xử lý"],
    ["SALES_EMPLOYEE", "Inconsistency/type", len(df), "Chuẩn hóa text; ID string; experience Int64"],
    ["SALES_EMPLOYEE", "Validation", len(df), "PK, required columns và experience không âm"],
]
pd.DataFrame(log, columns=["table","step","rows_after","result"]).to_csv(REPORT_DIR / "sales_employee_log.csv", index=False)
print("Đã xuất:", SILVER_DIR / "SALES_EMPLOYEE.csv")

Đã xuất: ../silver_data/SALES_EMPLOYEE.csv
